# Sembolsüz x86-64 fonksiyon adlandırma — QLoRA

Bu notebook, `mlx-lm` sohbet biçimindeki veriyi Qwen2.5-Coder üzerinde 4-bit QLoRA ile eğitir. Kayıp yalnız assistant yanıtında hesaplanır; sonunda test kümesinde `lora/olc.py` ile aynı ad F1 ölçümü yapılır.

Colab'da önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin. A100 40 GB varsa 7B, L4 24 GB varsa 3B model otomatik seçilir.

## 1. Sabit sürümlü bağımlılıklar

In [ ]:
%pip install -q \
  "transformers==4.48.3" \
  "peft==0.14.0" \
  "trl==0.15.2" \
  "bitsandbytes==0.45.2" \
  "accelerate==1.3.0" \
  "datasets==3.2.0"

## 2. Ayarlar ve GPU'ya göre model seçimi

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import zipfile

import torch

# "yukleme": veri.zip dosyasını Colab'ın sol dosya panelinden /content'e yükleyin.
# "drive": DRIVE_ZIP yolundaki arşivi kullanın.
VERI_KAYNAGI = "yukleme"                 # "yukleme" veya "drive"
YUKLENEN_ZIP = "/content/veri.zip"
DRIVE_ZIP = "/content/drive/MyDrive/asm-adlandirma/veri.zip"

# Açıkça başka model yazılabilir. Otomatik seçim açıkken belleği 35 GB'tan az
# GPU'larda bunun 3B karşılığı seçilir.
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # yol haritası: 1.5B; 7B için bu satırı değiştirip alttakini True yapın
BELLEGE_GORE_MODEL_SEC = False
L4_MODELI = "Qwen/Qwen2.5-Coder-3B-Instruct"

MAX_UZUNLUK = 3072
ETKIN_BATCH = 16
MIKRO_BATCH = 2
TOHUM = 7

if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadı; Colab çalışma zamanını GPU olarak değiştirin.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Bu notebook bf16 destekli A100 veya L4 bekliyor.")

gpu = torch.cuda.get_device_properties(0)
bellek_gb = gpu.total_memory / 1024**3
SECILEN_MODEL = MODEL
if BELLEGE_GORE_MODEL_SEC and bellek_gb < 35:
    SECILEN_MODEL = L4_MODELI

if ETKIN_BATCH % MIKRO_BATCH:
    raise ValueError("ETKIN_BATCH, MIKRO_BATCH değerine tam bölünmeli.")
BIRIKIM_ADIMI = ETKIN_BATCH // MIKRO_BATCH

print(f"GPU: {gpu.name} ({bellek_gb:.1f} GB)")
print(f"Model: {SECILEN_MODEL}")
print(f"Etkin batch: {MIKRO_BATCH} × {BIRIKIM_ADIMI} = {ETKIN_BATCH}")

## 3. Veriyi yükleme veya Drive'dan okuma

In [ ]:
if VERI_KAYNAGI == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    zip_yolu = Path(DRIVE_ZIP)
elif VERI_KAYNAGI == "yukleme":
    zip_yolu = Path(YUKLENEN_ZIP)
else:
    raise ValueError("VERI_KAYNAGI yalnız 'yukleme' veya 'drive' olabilir.")

if not zip_yolu.is_file():
    raise FileNotFoundError(f"Arşiv bulunamadı: {zip_yolu}")

calisma_dizini = Path("/content/asmsense")
acma_dizini = calisma_dizini / "veri-acilmis"
acma_dizini.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_yolu) as z:
    z.extractall(acma_dizini)

# zip_hazirla.sh dosyaları arşiv köküne koyar; yaygın iki alternatif düzen de kabul edilir.
adaylar = [
    acma_dizini,
    acma_dizini / "veri",
    acma_dizini / "lora" / "veri-15b-aciklama",
]
VERI_DIZINI = next(
    (p for p in adaylar if all((p / f"{ad}.jsonl").is_file() for ad in ("train", "valid", "test"))),
    None,
)
if VERI_DIZINI is None:
    raise FileNotFoundError("Arşivde train.jsonl, valid.jsonl ve test.jsonl birlikte bulunamadı.")

for ad in ("train", "valid", "test"):
    yol = VERI_DIZINI / f"{ad}.jsonl"
    with yol.open(encoding="utf-8") as f:
        adet = sum(1 for satir in f if satir.strip())
    print(f"{ad:>5}: {adet:>6} satır  ({yol})")

## 4. 4-bit model ve tüm lineer katmanlarda LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(TOHUM)
tokenizer = AutoTokenizer.from_pretrained(SECILEN_MODEL, use_fast=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

niceleme = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    SECILEN_MODEL,
    quantization_config=niceleme,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(
    model,
    LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    ),
)
model.print_trainable_parameters()

## 5. Sohbet şablonu ve yalnız assistant tokenlarında kayıp

Qwen sohbet şablonuyla kullanıcı istemi ve tam konuşma ayrı kodlanır. İsteme karşılık gelen etiketler `-100` yapıldığı için sistem ve kullanıcı tokenları kayba katılmaz.

In [ ]:
from datasets import load_dataset

ham_veri = load_dataset(
    "json",
    data_files={
        "train": str(VERI_DIZINI / "train.jsonl"),
        "validation": str(VERI_DIZINI / "valid.jsonl"),
    },
)

def kodla(ornek):
    mesajlar = ornek["messages"]
    if [m["role"] for m in mesajlar] != ["system", "user", "assistant"]:
        raise ValueError("Beklenen mesaj sırası: system, user, assistant")

    istem = tokenizer.apply_chat_template(
        mesajlar[:2], tokenize=True, add_generation_prompt=True
    )
    tumu = tokenizer.apply_chat_template(
        mesajlar, tokenize=True, add_generation_prompt=False
    )
    if tumu[:len(istem)] != istem:
        raise ValueError("Sohbet şablonunda assistant başlangıcı belirlenemedi.")

    tumu = tumu[:MAX_UZUNLUK]
    etiketler = [-100] * min(len(istem), len(tumu)) + tumu[len(istem):]
    if not any(x != -100 for x in etiketler):
        raise ValueError("Yanıt MAX_UZUNLUK dışında kaldı; veri hazırlama tavanını düşürün.")
    return {
        "input_ids": tumu,
        "attention_mask": [1] * len(tumu),
        "labels": etiketler,
        "uzunluk": len(tumu),
    }

tokenli_veri = ham_veri.map(
    kodla,
    remove_columns=ham_veri["train"].column_names,
    desc="Qwen sohbet şablonu uygulanıyor",
)

for bolum in ("train", "validation"):
    uzunluklar = tokenli_veri[bolum]["uzunluk"]
    print(
        f"{bolum:>10}: {len(uzunluklar)} örnek; "
        f"ortalama {sum(uzunluklar) / len(uzunluklar):.0f}, en uzun {max(uzunluklar)} token"
    )
tokenli_veri = tokenli_veri.remove_columns("uzunluk")

## 6. İki epoch eğitim

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

egitim_ayari = TrainingArguments(
    output_dir="/content/asmsense/denetim-noktalari",
    num_train_epochs=2.0,
    per_device_train_batch_size=MIKRO_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=BIRIKIM_ADIMI,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    seed=TOHUM,
    data_seed=TOHUM,
    group_by_length=True,
    remove_unused_columns=False,
)

veri_toplayici = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
egitici = Trainer(
    model=model,
    args=egitim_ayari,
    train_dataset=tokenli_veri["train"],
    eval_dataset=tokenli_veri["validation"],
    data_collator=veri_toplayici,
)
egitim_sonucu = egitici.train()
egitici.evaluate()
egitim_sonucu

## 7. Adaptörü Drive'a kaydetme ve zip olarak indirme

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
model_kisa = SECILEN_MODEL.rsplit("/", 1)[-1]
ADAPTOR_DIZINI = Path("/content/drive/MyDrive/asm-adlandirma") / f"adaptor-{model_kisa}"
ADAPTOR_DIZINI.parent.mkdir(parents=True, exist_ok=True)

egitici.save_model(str(ADAPTOR_DIZINI))
tokenizer.save_pretrained(str(ADAPTOR_DIZINI))

zip_koku = Path("/content") / ADAPTOR_DIZINI.name
zip_yolu = Path(shutil.make_archive(
    str(zip_koku),
    "zip",
    root_dir=ADAPTOR_DIZINI.parent,
    base_dir=ADAPTOR_DIZINI.name,
))
print(f"Drive: {ADAPTOR_DIZINI}")
print(f"İndirilecek arşiv: {zip_yolu} ({zip_yolu.stat().st_size / 1024**2:.0f} MB)")
files.download(str(zip_yolu))

## 8. Test ölçümü

Üretim greedy'dir (`do_sample=False`) ve en çok 160 yeni token kullanır. Sonuç satırları `lora/olc.py` biçimindedir.

In [ ]:
import gc
from tqdm.auto import tqdm

# lora/olc.py, taban.py içindeki bu iki fonksiyonu kullanır; tanım birebir kopyadır.
def kelimeler(ad: str) -> list[str]:
    ad = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", ad)
    return [k for k in re.split(r"[_\W]+", ad.lower()) if k]


def f1(tahmin: str, gercek: str) -> float:
    t, g = kelimeler(tahmin), kelimeler(gercek)
    ortak = len(set(t) & set(g))
    if not ortak:
        return 0.0
    p, r = ortak / len(set(t)), ortak / len(set(g))
    return 2 * p * r / (p + r)


del egitici
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = True
model.eval()

test_satirlari = []
with (VERI_DIZINI / "test.jsonl").open(encoding="utf-8") as f:
    test_satirlari = [json.loads(satir) for satir in f if satir.strip()]

sonuclar = []
for kayit in tqdm(test_satirlari, desc="Greedy test üretimi"):
    mesajlar = kayit["messages"][:2]
    girdi = tokenizer.apply_chat_template(
        mesajlar,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    if girdi.shape[1] > MAX_UZUNLUK:
        raise ValueError(f"Test girdisi {MAX_UZUNLUK} tokenı aşıyor: {kayit.get('id')}")

    with torch.inference_mode():
        uretilen = model.generate(
            input_ids=girdi,
            attention_mask=torch.ones_like(girdi),
            max_new_tokens=160,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    yeni_tokenlar = uretilen[0, girdi.shape[1]:]
    metin = tokenizer.decode(yeni_tokenlar, skip_special_tokens=True)
    eslesmeler = re.findall(r"\{[^{}]*\}", metin)
    try:
        cevap = json.loads(eslesmeler[-1]) if eslesmeler else {"ad": metin.strip()[:60]}
    except json.JSONDecodeError:
        cevap = {}

    tahmin = str(cevap.get("ad", ""))
    aciklama = str(cevap.get("aciklama", ""))
    ogretmen = json.loads(kayit["messages"][2]["content"])
    gercek = str(ogretmen["ad"])
    skor = f1(tahmin, gercek)
    sonuc = {
        "id": kayit.get("id", ""),
        "gercek": gercek,
        "tahmin": tahmin,
        "aciklama": aciklama,
        "ogretmen": str(ogretmen.get("aciklama", "")),
        "f1": round(skor, 3),
        "opt": kayit.get("opt", ""),
        "token": int(girdi.shape[1] + yeni_tokenlar.shape[0]),
    }
    sonuclar.append(sonuc)
    print(f"{skor:.2f}  {sonuc['opt']}  {gercek:<28} ← {tahmin}")

sonuc_yolu = Path("/content") / f"sonuc-{model_kisa}-lora.jsonl"
with sonuc_yolu.open("w", encoding="utf-8") as f:
    for sonuc in sonuclar:
        f.write(json.dumps(sonuc, ensure_ascii=False) + "\n")

def ozet_yaz(etiket, satirlar):
    if not satirlar:
        print(f"{etiket}: örnek yok")
        return
    ortalama = sum(r["f1"] for r in satirlar) / len(satirlar)
    tam = sum(r["f1"] == 1 for r in satirlar)
    print(f"{etiket}: ortalama F1 {ortalama:.3f}  (n={len(satirlar)}, tam isabet {tam})")

for opt in ("-O0", "-O2"):
    ozet_yaz(opt, [r for r in sonuclar if r["opt"] == opt])
ozet_yaz("genel", sonuclar)
print(f"ayrıntı → {sonuc_yolu}")
files.download(str(sonuc_yolu))